# DL & GenAI project milestone - 3 (23f3001252)

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


## Load datasets

In [2]:
## Load datasets
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sample = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

In [3]:
print("Train Shape:", train_df.shape)
print("Test Shape:", test_df.shape)
print("Sample Shape:", sample.shape)

Train Shape: (2000, 8)
Test Shape: (500, 7)
Sample Shape: (500, 2)


In [4]:
train_df.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [5]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


In [6]:
!pip install faiss-cpu #install FAISS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 83.3 MB/s eta 0:00:00:00:0100:01


In [7]:
import pandas as pd 
import numpy as np 
import faiss 
from sentence_transformers import SentenceTransformer, CrossEncoder 
from transformers import AutoTokenizer, pipeline 
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.metrics.pairwise import cosine_similarity 

In [8]:
print("Creating nowledge base")
kb = [] 
for idx, row in train_df.iterrows(): 
    correct_letter = row['answer'] 
    kb.append(str(row[correct_letter])) 

print("Loading embedding model and creating index") 
model = SentenceTransformer('all-MiniLM-L6-v2') 
kb_embeddings = model.encode(kb, show_progress_bar=False) 
index = faiss.IndexFlatL2(kb_embeddings.shape[1]) 
index.add(kb_embeddings)

print("Knowledge base successfully created")

Creating nowledge base
Loading embedding model and creating index


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge base successfully created


In [9]:
## Zero-shot classifier for Q1, Q2, Q6
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli") 
row_150 = train_df.iloc[150] 
prompt_150 = str(row_150['prompt']) 
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])] 
ans_150 = str(row_150[row_150['answer']])

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## Question 1
#### Q1. Run the zero-shot classifier on facebook/bart-large-mnli on prompt for the row index 150. Pass the 5 options (A-E) candidate_labels. What is the predicted probability score assigned to the ground-truth correct option (option in the answer column)? (Round to 3 decimal points)

In [10]:
# Q1

result = zs(
    prompt_150,
    candidate_labels=labels_150,
    multi_label=False
)

# Convert to dictionary
scores = dict(zip(result["labels"], result["scores"]))

# Probability assigned to the correct answer
gt_score = scores[ans_150]

print("Ground Truth Answer:", ans_150)
print("Probability:", round(gt_score, 3))

Ground Truth Answer: The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
Probability: 0.384


## Question 2
#### Q2. Embed the prompt for row index 150 using all-MiniLM-L6-v2. Query your FAISS index to retrieve the top k=10 most similar documents. At what exact rank (1 through 10) did FAISS place the true correct document (which is the document originally located at index 150 in the KB)?

In [11]:
# Q2

# Embed the prompt
query_embedding = model.encode([prompt_150])

# Search Top-10
D, I = index.search(query_embedding, k=10)

print("Retrieved KB indices:", I[0])

# Find the rank of the correct document (KB index 150)
if 150 in I[0]:
    rank = list(I[0]).index(150) + 1
    print("Rank of correct document:", rank)
else:
    print("Correct document not found in Top-10")

Retrieved KB indices: [ 663 1701 1269 1532  576  847 1693 1906  168  150]
Rank of correct document: 10


## Question - 3
#### Q3. Take the top 10 documents retrieved by FAISS in the previous question. Load cross-encoder/ms-marco-MiniLM-L-6-v2. Score the prompt against these 10 documents and sort them by the cross-encoder's score. At what exact rank (1 through 10) does the Cross-Encoder place the true correct document?

In [14]:
# Q3

# Get top-10 retrieved indices from FAISS
query_embedding = model.encode([prompt_150])
D, I = index.search(query_embedding, k=10)

retrieved_indices = I[0]

# Load Cross-Encoder
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# Retrieve top-10 documents
docs_10 = [kb[i] for i in retrieved_indices]

# Create (question, document) pairs
pairs = [[prompt_150, doc] for doc in docs_10]

# Score each pair
ce_scores = cross_encoder.predict(pairs)

# Sort by score (highest first)
reranked = sorted(
    zip(retrieved_indices, ce_scores),
    key=lambda x: x[1],
    reverse=True
)

# Find rank of true document (KB index 150)
for rank, (idx, score) in enumerate(reranked, start=1):
    if idx == 150:
        print("Cross-Encoder Rank:", rank)
        break

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-Encoder Rank: 1


## Question - 4
Q4. Retrieve the top k=5 documents for the prompt at row index 42. Concatenate them with a single space between each. Create a string: "Context: [concatenated_docs] Question: [prompt]". Tokenize this string using the bert-base-uncased tokenizer (without truncation). Exactly how many total tokens does this generate?

In [16]:
# Q4

row_42 = train_df.iloc[42]
prompt_42 = str(row_42["prompt"])

# Embed prompt
query_embedding = model.encode([prompt_42])

# Retrieve Top-5 documents
D, I = index.search(query_embedding, k=5)

docs_5 = [kb[i] for i in I[0]]

# Concatenate with one space
context = " ".join(docs_5)

rag_string = f"Context: {context} Question: {prompt_42}"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

tokens = tokenizer(
    rag_string,
    truncation=False
)

print("Total Tokens:", len(tokens["input_ids"]))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Total Tokens: 216


## Question - 5

Q5. Retrieve the exact true document for row index 150 from your KB. Create a RAG string: "Context: [true_document] Question: [prompt]". Run the same zero-shot classification from Question 1 on this augmented string. What is the new predicted probability score of the ground-truth correct option? (Round to 3 decimal places).

In [17]:
# Q5

# True document
true_document = kb[150]

# Create RAG prompt
rag_prompt = f"Context: {true_document} Question: {prompt_150}"

# Run zero-shot classifier
result = zs(
    rag_prompt,
    candidate_labels=labels_150,
    multi_label=False
)

scores = dict(zip(result["labels"], result["scores"]))

print("Ground Truth Answer:", ans_150)
print("New Probability:", round(scores[ans_150], 3))

Ground Truth Answer: The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
New Probability: 0.989


## Question - 6

Q6. What happens if your vector database retrieves the wrong information? Take the prompt for row index 150. Manually force the context to be the document located at KB index 999 (a completely unrelated fact). Run the zero-shot classifier on this "Adversarial RAG" string. What is the probability of the correct option now? (Round to 3 decimal places).

In [18]:
# Q6

# Use an unrelated document as context
wrong_document = kb[999]

# Create adversarial RAG prompt
adv_prompt = f"Context: {wrong_document} Question: {prompt_150}"

# Run zero-shot classifier
result = zs(
    adv_prompt,
    candidate_labels=labels_150,
    multi_label=False
)

scores = dict(zip(result["labels"], result["scores"]))

print("Correct Answer:", ans_150)
print("Probability:", round(scores[ans_150], 3))

Correct Answer: The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
Probability: 0.529


## Question - 7

In RAG, a "Hit" occurs if the retrieved context contains the facts needed to answer the question.

Q7. For the first 100 rows of train.csv (indices 0-99), retrieve the top k=5 documents for each prompt. If the exact string of the row's correct option is found inside any of those 5 retrieved documents, it counts as a hit. What is the exact Hit Rate percentage (0 to 100) for these 100 rows? (Round to 1 decimal place).

In [20]:
# Q7

hits = 0
total = 100

for idx in range(total):

    row = train_df.iloc[idx]

    prompt = str(row["prompt"])

    correct_doc = str(row[row["answer"]])

    # Embed prompt
    query_embedding = model.encode([prompt])

    # Retrieve Top-5
    D, I = index.search(query_embedding, k=5)

    retrieved_docs = [kb[i] for i in I[0]]

    # Hit if exact correct document appears
    if correct_doc in retrieved_docs:
        hits += 1

hit_rate = hits / total * 100

print("Hits:", hits)
print("Hit Rate (%):", round(hit_rate, 1))

Hits: 73
Hit Rate (%): 73.0


## Question - 8

Q8. Build a loop that processes the first 20 rows (indices 0 through 19) of train.csv.
For each row, your pipeline must do the following in order:

Retrieve: Embed the prompt and retrieve the top k=5 documents from your FAISS Knowledge Base.

Rerank: Pass the prompt and those 5 documents into the ms-marco-MiniLM-L-6-v2 Cross-Encoder. Select the single document with the highest cross-encoder score.

Augment: Create your RAG string exactly formatted as: "Context: [best_document] Question: [prompt]".

Predict: Pass this augmented string to the facebook/bart-large-mnli zero-shot classifier, using the 5 options (A, B, C, D, E) as your candidate_labels.

Score: Look at the probability scores output by the model. Rank the options from highest probability to lowest. Take the top 3 letters (e.g., ['C', 'A', 'E']) and calculate the MAP@3 for that row.

What is the final average MAP@3 score of this state-of-the-art RAG pipeline across these 20 rows? (Round to 3 decimal places).

In [21]:
# Q8

cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def apk(actual, predicted, k=3):
    """
    Average Precision at K
    """
    if len(predicted) > k:
        predicted = predicted[:k]

    score = 0.0

    for i, p in enumerate(predicted):
        if p == actual:
            score = 1.0 / (i + 1)
            break

    return score


map_scores = []

for idx in range(20):

    row = train_df.iloc[idx]

    prompt = str(row["prompt"])

    labels = [
        str(row["A"]),
        str(row["B"]),
        str(row["C"]),
        str(row["D"]),
        str(row["E"])
    ]

    answer_letter = row["answer"]

    
    # Retrieve Top-5
    query_embedding = model.encode([prompt])

    D, I = index.search(query_embedding, k=5)

    retrieved_indices = I[0]

    docs = [kb[i] for i in retrieved_indices]

    
    # Cross Encoder Reranking
    pairs = [[prompt, doc] for doc in docs]

    scores = cross_encoder.predict(pairs)

    best_doc = docs[np.argmax(scores)]

    
    # RAG Prompt
    rag_prompt = f"Context: {best_doc} Question: {prompt}"


    # Zero-shot Prediction
    result = zs(
        rag_prompt,
        candidate_labels=labels,
        multi_label=False
    )

    ranked_labels = result["labels"]

    # Convert option text -> option letter
    option_map = {
        str(row["A"]): "A",
        str(row["B"]): "B",
        str(row["C"]): "C",
        str(row["D"]): "D",
        str(row["E"]): "E"
    }

    predicted_letters = [option_map[label] for label in ranked_labels[:3]]

    map_scores.append(
        apk(answer_letter, predicted_letters, k=3)
    )

final_map3 = np.mean(map_scores)

print("Final MAP@3:", round(final_map3, 3))

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Final MAP@3: 0.975
